In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from pathlib import Path
import spacy as sp
import spacy-

In [ ]:
input_path=Path().resolve()/'input'

# Loading Dataa

In [ ]:
training_data=pl.read_csv(input_path/'Corona_NLP_train.csv')

# Data Preprocessing

### Data Preprocessing pipeline:
1. clean missing values
2. fix data types
3. Text Cleaning
4. tokenize the text of the target column
5. remove stop words
6. Lemmatize the words using pos tagging
7. reconstruct the row from the tokenize words

In [ ]:
# checking nulls
training_data.null_count()

null exists at location

We will impute those nulls with unknown value

In [ ]:
training_data=training_data.with_columns(pl.col('Location').fill_null('unknown'))
training_data.null_count()

In [ ]:
# Converting the tweetat to polars datetime
training_data=training_data.with_columns(pl.col('TweetAt').str.to_date())

In [ ]:
# Tokenize words and Clean Text
nlp=sp.load('en_core_web_sm')
text=nlp("@MeNyrbie @Phil_Gahan @Chrisit…")
for word in text:
    print(word.is_oov)

# Data Exploration

In [ ]:
training_data.head(10)

In [ ]:
training_data.describe()

There are many null values in the location field <br>
Date format for the TweetAt is not accurate needs to be changed

In [ ]:
## understanding screen name
training_data.select(pl.col('ScreenName')).unique().count()

In [ ]:
# All values are unique so its an identity to the test

In [ ]:
# describing the categorical variables
training_data.to_pandas().describe(include="object")

- Most common location is Londan 
- Most common tweet date is 3448
- Most common Sentiment is Positive

In [ ]:
# Visualizing the sentiment column


In [ ]:
from lib import loadProfile
sentiment_data=training_data.group_by('Sentiment').len().select(pl.col('Sentiment'), pl.col('len').alias('Counts'))

In [ ]:
Plots=loadProfile(sentiment_data)
Plots.barplot_seaborn('Sentiment','Counts')

In [ ]:
Plots.pie(sentiment_data,sentiment_data['Counts'],sentiment_data['Sentiment'])

In [ ]:
# Correlation between Sentiment and Original Tweet
data_corr=training_data.select(pl.corr('OriginalTweet','Sentiment',method="spearman"))
data_corr